## Test Grid SANOS



In [1]:
%load_ext autoreload
%autoreload 1

from quantdata.dysanos.simple.aec import load_data
data = load_data(["SPX"], reduced=True).data['SPX']
print(f"/done; columns {data.keys()}")



00: Importing torch...done; this took 1.4s.
01:   Generating daily data for SPX time series from 2025-04-01 to 2025-07-01 excluding half days:
02:     Options filter: nopt=1000,bdte=756,nms=4.0,lvolu=1.0,loi=1.0,rmam=False,vegaSV=0.001,mxpp=10
02:     MkArbFree config: surface,mhs=0.0001,mw=0.10,mik=5,mdn=0.0001,dC=0.001,dps=0.0001
02:     Raw Day filter: bid=0.011
02:     Expiry filter: nat,eth=10,bth=4,mps=1e-06,miv=0.01,xiv=5.0
02:     Pool config: workers 4p
01:   Generated daily filtered options data 63 dates for SPX from 2025-04-01 to 2025-07-01 excluding half days. Maximum number of options is 1000. This took 19:33.
00: Aggregating arbitrage-free data for ticker SPX for 63 business days from 2025-04-01 to 2025-07-01 excluding half days:
01:   Options filter: nopt=1000,bdte=756,nms=4.0,lvolu=1.0,loi=1.0,rmam=False,vegaSV=0.001,mxpp=10
01:   MkArbFree config: surface,mhs=0.0001,mw=0.10,mik=5,mdn=0.0001,dC=0.001,dps=0.0001
01:   Raw Day filter: bid=0.011
01:   Expiry filter: nat,et

In [ ]:
%aimport quantdata.utils.arbfree

from cdxcore.bs import bs
from cdxcore.dynaplot import figure, store, m_o_m
from cdxcore.verbose import Context
from cdxcore.pretty import PrettyObject
from quantdata.biz_calendar import DAYS_PER_YEAR
from quantdata.utils.arbfree import MkArbFree
from dataclasses import dataclass, field
import math
import numpy as np
import time

def fit(data):
    print(data.keys()   )
    num_samples = len(data.dates)

    mkArbFree = MkArbFree("surface", smoothness=0.01, mid_fit_weight=0.1 )

    verbose = Context.all

    ticker = "SPX"
    afs    = {}
    for isample in range(num_samples-1,max(-1,num_samples-200),-20):
        d = data.dates[isample]
        verbose.write(f"{ticker} {d}:")        

        is_valid = data.is_valid[isample,:]
        uttys = np.unique(data.tte_y[isample,:][is_valid])
        r = PrettyObject()
        r.expiries = []
        r.strikes = []
        r.bids = []
        r.asks = []

        for utty in uttys:
            mask = (data.tte_y[isample,:] == utty) & is_valid
            r.expiries.append(utty)
            r.strikes.append(data.pure_strike[isample][mask])
            r.bids.append(data.pure_bid_price[isample][mask])
            r.asks.append(data.pure_ask_price[isample][mask])
            ixs = np.argsort(r.strikes[-1])
            r.strikes[-1] = r.strikes[-1][ixs]
            r.bids[-1] = r.bids[-1][ixs]
            r.asks[-1] = r.asks[-1][ixs]
        
        af = mkArbFree( expiries=r.expiries, strikes=r.strikes, bids=r.bids, asks=r.asks,
                        verbose=verbose(1) )

        afs[d] = af

fit(data)


dict_keys(['ticker', 'dates', 'is_valid', 'time_y', 'ask_vol', 'bdte', 'bid_vol', 'call_ask', 'call_bid', 'call_opt_id', 'df', 'expiry_id', 'fit1_vol', 'fitX_vol', 'fwd', 'fwd_ref', 'is_am', 'is_american', 'log_open_interest', 'log_volume', 'mask_arb_free', 'mkt_call_ask', 'mkt_call_bid', 'mkt_call_log_open_interest', 'mkt_call_log_volume', 'mkt_put_ask', 'mkt_put_bid', 'mkt_put_log_open_interest', 'mkt_put_log_volume', 'nms', 'pure_ask_delta', 'pure_ask_dk', 'pure_ask_gamma', 'pure_ask_price', 'pure_ask_theta', 'pure_ask_vega', 'pure_bid_delta', 'pure_bid_dk', 'pure_bid_gamma', 'pure_bid_price', 'pure_bid_theta', 'pure_bid_vega', 'pure_fit1_delta', 'pure_fit1_dk', 'pure_fit1_gamma', 'pure_fit1_price', 'pure_fit1_theta', 'pure_fit1_vega', 'pure_fitX_delta', 'pure_fitX_dk', 'pure_fitX_gamma', 'pure_fitX_price', 'pure_fitX_theta', 'pure_fitX_vega', 'pure_strike', 'put_opt_id', 'spot_prev', 'spot_ref', 'spot_ret', 'strike', 'tte_y'])
00: SPX 2025-07-01:
01:   Expiry #0 1.0d: Generating ex